# PyTorch on Apple MPS

Master GPU acceleration on Apple Silicon with Metal Performance Shaders. Learn device setup, tensor dtypes, and the critical fp64 trap that trips up newcomers.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/pytorch-mps-basics)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Colab provides PyTorch with CUDA support, so these notebooks run on a real GPU rather than Apple's MPS backend. If a code sample uses `device = torch.device('mps' ...)`, it will fall back to `cpu` on Colab unless you adapt it; replace `'mps'` with `'cuda'` for GPU execution.


In [ ]:
!pip install torch numpy

---

## Lesson examples (optional)

These are the runnable code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.


### Lesson example: Device Setup & MPS Basics


In [ ]:
import torch
import time

# 1. Check MPS availability
print(f"MPS available: {torch.backends.mps.is_available()}")

# 2. Create device-agnostic setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 3. Create and move tensors
x = torch.randn(1000, 1000, device=device)
print(f"Tensor device: {x.device}, dtype: {x.dtype}")

# 4. Device transfer with .to()
model = torch.nn.Linear(100, 10)
model = model.to(device)
print(f"Model moved to: {next(model.parameters()).device}")

# 5. Avoid fp64 on MPS (demonstrate error gracefully)
try:
    bad_tensor = torch.randn(10, dtype=torch.float64, device=device)
except RuntimeError as e:
    print(f"Error (expected on MPS): {str(e)[:60]}...")

# 6. Always use float32 on MPS
good_tensor = torch.randn(10, dtype=torch.float32, device=device)
print(f"Created fp32 tensor: {good_tensor.shape}, dtype: {good_tensor.dtype}")

# 7. Benchmark: simple matmul
x = torch.randn(2000, 2000, device=device)
y = torch.randn(2000, 2000, device=device)
if device.type == "mps":
    torch.mps.synchronize()
start = time.perf_counter()
z = x @ y
if device.type == "mps":
    torch.mps.synchronize()
end = time.perf_counter()
print(f"Matmul (2000x2000) took {(end-start)*1000:.2f}ms")

### Lesson example: Tensors, Dtypes, and the fp64 Trap


In [ ]:
import torch
import torch.nn as nn

# 1. Tensor properties
x = torch.randn(2, 3, 4)
print(f"Shape: {x.shape}, Dtype: {x.dtype}, Device: {x.device}, Requires grad: {x.requires_grad}")

# 2. Create tensors with specific dtypes
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
a = torch.randn(5, dtype=torch.float32, device=device)
b = torch.randn(5, dtype=torch.bfloat16, device=device)
c = torch.randn(5, dtype=torch.float16, device=device)

print(f"\nDtype comparison:")
print(f"float32: {a.dtype}, bfloat16: {b.dtype}, float16: {c.dtype}")

# 3. Demonstrate float16 underflow with small numbers
x_fp16 = torch.tensor(1e-5, dtype=torch.float16)
x_bf16 = torch.tensor(1e-5, dtype=torch.bfloat16)
x_fp32 = torch.tensor(1e-5, dtype=torch.float32)
print(f"\nSmall value (1e-5) representation:")
print(f"float16: {x_fp16.item():.10e}")
print(f"bfloat16: {x_bf16.item():.10e}")
print(f"float32: {x_fp32.item():.10e}")

# 4. Create model and check dtype
model = nn.Linear(10, 5).float().to(device)
print(f"\nModel on device: {next(model.parameters()).device}")
print(f"Model dtype: {next(model.parameters()).dtype}")

# 5. The fp64 trap (commented out to avoid crash on MPS)
print("\nfp64 trap: Uncomment below to test")
# bad_tensor = torch.randn(10, dtype=torch.float64, device=device)
# RuntimeError on MPS: float64 is not supported

# 6. Safe dtype conversion pattern
model = nn.Linear(10, 5).to(device).float()
print(f"Safe model dtype: {next(model.parameters()).dtype}")

# 7. Mixed precision setup example
if device.type == "mps" or device.type == "cuda":
    scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None
    autocast_dtype = torch.bfloat16 if device.type == "mps" else torch.float16
    print(f"\nUsing mixed precision: autocast with {autocast_dtype}")

# 8. Tensor creation with explicit device
x = torch.randn(100, 50, device=device, dtype=torch.float32)
print(f"\nCreated tensor: shape {x.shape}, device {x.device}, dtype {x.dtype}")

---

## Exercise: MPS Device Benchmark


Create tensors on CPU and MPS, benchmark matrix multiplication at increasing sizes, and report the speedup factor.

**Requirements:**
- Create square matrices of sizes: 256, 512, 1024, 2048
- For each size, benchmark matmul on both CPU and MPS
- Use `torch.mps.synchronize()` before measuring MPS time (ensures all GPU ops complete)
- Report speedup as CPU_time / MPS_time for each size
- Handle the case where MPS is unavailable gracefully
- Use consistent number of iterations (5) for all sizes

**Hints:**
- Use `time.perf_counter()` for timing (more precise than `time.time()`)
- Warm up the GPU with one iteration before timing
- Run at least 5 iterations per size to reduce noise
- MPS synchronization is critical; without it, timers fire before GPU finishes
- Print results in a clear table format for easy comparison


In [ ]:
import torch
import time

device_mps = torch.device("mps" if torch.backends.mps.is_available() else None)
device_cpu = torch.device("cpu")

def benchmark_matmul(size, device, num_iters=5):
    """Benchmark matrix multiplication on a given device."""
    # Create random matrices
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)

    # Warm up (1 iteration)
    _ = torch.matmul(a, b)
    if device.type == "mps":
        torch.mps.synchronize()

    # Timed iterations
    start = time.perf_counter()
    for _ in range(num_iters):
        _ = torch.matmul(a, b)
    if device.type == "mps":
        torch.mps.synchronize()
    end = time.perf_counter()

    return (end - start) / num_iters

# TODO: Test matmul at sizes [256, 512, 1024, 2048]
# TODO: Compute speedup (CPU time / MPS time) for each size
# TODO: Print results in a table with columns: Size, CPU (ms), MPS (ms), Speedup (x)
# TODO: Handle case where MPS is None

---

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/pytorch-mps-basics) for the solution and discussion.
